# 03 — Chapter A data preparation (Descriptive Clustering)

**Project:** Network Traffic Clustering and Novelty Detection
**Reference:** *Project Proposal V2*, §5 (Chapter A) and Phase 3.

**Purpose.** Turn the *whole* official training split (mixed normal + attack —
Chapter A is descriptive) and the official testing split into finite, scaled
feature matrices, using the representation decisions frozen in
`02_feature_representation.ipynb`. **No clustering is performed here.**

**Leakage discipline.**
* Every fitted transformer is fit **only on the Chapter-A training split**.
* The testing split is transformed with the already-fitted transformers.
* `label` and `attack_cat` are saved **separately** and never placed in a feature
  matrix — they exist only for the later external audit.

Three representations are produced (from notebook 02): `ohe_full` (baseline),
`ohe_topk` (primary), `freq` (alternative).

## 0. Environment setup

In [1]:
import sys, joblib
from pathlib import Path

def _find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook inside the project repository.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from config.config import (
    CHAPTER_A_DIR, PROCESSED_DIR, REPORTS_DIR, METRICS_DIR,
    RANDOM_SEED, FEATURE_DICT_FILE, ID_COLUMNS, LABEL_COLUMN, ATTACK_CAT_COLUMN,
)
from src.data.preprocess import (
    load_official_datasets, split_features_labels, check_finite, FeaturePipeline,
)

RANDOM_STATE = RANDOM_SEED
np.random.seed(RANDOM_STATE)
CHAPTER_A_DIR.mkdir(parents=True, exist_ok=True)
print("Chapter A output dir:", CHAPTER_A_DIR)

Chapter A output dir: /home/aminelby/Network-Traffic-Clustering/data/processed/chapter_a


## 1. Load the official splits and separate labels

**Purpose.** Rebuild the data from zero (no reliance on other notebooks) and fix
the roles: features vs reference labels. Identifiers (`id`) are dropped.

In [2]:
train_df, test_df = load_official_datasets()
X_train_raw, y_train = split_features_labels(train_df)
X_test_raw, y_test = split_features_labels(test_df)

print("Chapter A train features:", X_train_raw.shape)
print("Chapter A test  features:", X_test_raw.shape)
print("labels train:", y_train.shape, "| labels test:", y_test.shape)

# Guard: no label / identifier column may be present in the feature frames.
forbidden = set(ID_COLUMNS + [LABEL_COLUMN, ATTACK_CAT_COLUMN])
assert not (forbidden & set(X_train_raw.columns)), "Label/id leaked into features!"
print("\n[OK] Feature frames contain no id / label / attack_cat columns.")

Chapter A train features: (82332, 42)
Chapter A test  features: (175341, 42)
labels train: (82332, 2) | labels test: (175341, 2)

[OK] Feature frames contain no id / label / attack_cat columns.


## 2. Fit the three representations on training data only

**Purpose.** `FeaturePipeline` fits its `log1p` list, its `StandardScaler` and its
categorical encoder on `X_train_raw` **only** (the Chapter-A training split). The
test split is never seen during fitting.

In [3]:
representations = ["ohe_full", "ohe_topk", "freq"]
pipelines, matrices = {}, {}

for name in representations:
    pipe = FeaturePipeline(categorical_strategy=name).fit(X_train_raw)
    Xtr = pipe.transform(X_train_raw)
    Xte = pipe.transform(X_test_raw)
    pipelines[name], matrices[name] = pipe, (Xtr, Xte)
    print(f"{name:9s}: train {Xtr.shape} | test {Xte.shape} | "
          f"dtype {Xtr.dtype} | fitted on train only")

ohe_full : train (82332, 190) | test (175341, 190) | dtype float32 | fitted on train only


ohe_topk : train (82332, 63) | test (175341, 63) | dtype float32 | fitted on train only


freq     : train (82332, 42) | test (175341, 42) | dtype float32 | fitted on train only


## 3. Leakage and integrity checks

**Purpose.** Verify, for every representation, that: dimensions align between train
and test; the matrices are finite; columns are in the same order; no label/identifier
name appears in the feature names; and record sparsity (memory relevance).

In [4]:
audit_rows = []
for name in representations:
    pipe = pipelines[name]
    Xtr, Xte = matrices[name]
    feat_names = pipe.get_feature_names()

    check_finite(Xtr, f"{name} train")
    check_finite(Xte, f"{name} test")
    assert Xtr.shape[1] == Xte.shape[1] == len(feat_names), "column count mismatch"
    assert not (forbidden & set(feat_names)), "label/id name in feature matrix!"
    assert np.isfinite(Xtr).all() and np.isfinite(Xte).all()

    audit_rows.append({
        "representation": name,
        "n_features": Xtr.shape[1],
        "train_rows": Xtr.shape[0],
        "test_rows": Xte.shape[0],
        "dtype": str(Xtr.dtype),
        "train_zero_fraction": float((Xtr == 0).mean()),
        "test_zero_fraction": float((Xte == 0).mean()),
        "all_finite": True,
        "columns_aligned": True,
    })
audit = pd.DataFrame(audit_rows).set_index("representation")
display(audit.round(4))
audit.to_csv(METRICS_DIR / "03_chapter_a_audit.csv")
print("Saved ->", METRICS_DIR / "03_chapter_a_audit.csv")

,n_features,train_rows,test_rows,dtype,train_zero_fraction,test_zero_fraction,all_finite,columns_aligned
representation,,,,,,,,
ohe_full,190,82332,175341,float32,0.7789,0.7789,True,True
ohe_topk,63,82332,175341,float32,0.3333,0.3333,True,True
freq,42,82332,175341,float32,0.0000,0.0000,True,True


Saved -> /home/aminelby/Network-Traffic-Clustering/results/metrics/03_chapter_a_audit.csv


## 4. Save the processed matrices, labels and fitted preprocessors

**Purpose.** Persist exactly what later chapters need — the matrices (compressed
`float32`), the aligned reference labels, and the **fitted pipelines** (joblib) so
the identical transformations can be reused without re-fitting.

Note: no large sparse matrix is densified carelessly — the maximum matrix is
~175k × 190 `float32` ≈ 133 MB, and each representation is written one at a time.

In [5]:
for name in representations:
    Xtr, Xte = matrices[name]
    pipe = pipelines[name]

    np.savez_compressed(CHAPTER_A_DIR / f"X_train_{name}.npz", X=Xtr)
    np.savez_compressed(CHAPTER_A_DIR / f"X_test_{name}.npz", X=Xte)
    pd.Series(pipe.get_feature_names(), name="feature_name").to_csv(
        CHAPTER_A_DIR / f"feature_names_{name}.csv", index=False)
    joblib.dump(pipe, CHAPTER_A_DIR / f"pipeline_{name}.joblib")

    # free memory before the next representation
    del matrices[name]

# Reference labels (kept separate from the feature matrices).
y_train.to_csv(CHAPTER_A_DIR / "y_train.csv", index=False)
y_test.to_csv(CHAPTER_A_DIR / "y_test.csv", index=False)

print("Saved to", CHAPTER_A_DIR)
for p in sorted(CHAPTER_A_DIR.glob("*")):
    print(f"  {p.name:32s} {p.stat().st_size/1024:8.1f} KB")

Saved to /home/aminelby/Network-Traffic-Clustering/data/processed/chapter_a
  X_test_freq.npz                    6659.3 KB
  X_test_ohe_full.npz                8574.9 KB
  X_test_ohe_topk.npz                6989.5 KB
  X_train_freq.npz                   3141.6 KB
  X_train_ohe_full.npz               3981.3 KB
  X_train_ohe_topk.npz               3275.1 KB
  feature_names_freq.csv                0.4 KB
  feature_names_ohe_full.csv            2.1 KB
  feature_names_ohe_topk.csv            0.6 KB
  pipeline_freq.joblib                  9.7 KB
  pipeline_ohe_full.joblib              4.6 KB
  pipeline_ohe_topk.joblib              4.5 KB
  y_test.csv                         1719.7 KB
  y_train.csv                         788.6 KB


## 5. Reload-and-verify (round trip)

**Purpose.** Prove the artefacts on disk are complete: reload one representation and
its pipeline, re-transform the raw test frame, and confirm the result is
byte-identical to the saved matrix and has the expected shape.

In [6]:
name = "ohe_topk"
pipe_reloaded = joblib.load(CHAPTER_A_DIR / f"pipeline_{name}.joblib")
X_test_reloaded = np.load(CHAPTER_A_DIR / f"X_test_{name}.npz")["X"]
X_test_recomputed = pipe_reloaded.transform(X_test_raw)

assert X_test_reloaded.shape == X_test_recomputed.shape
assert np.allclose(X_test_reloaded, X_test_recomputed)
assert np.isfinite(X_test_reloaded).all()

names_reloaded = pd.read_csv(CHAPTER_A_DIR / f"feature_names_{name}.csv")["feature_name"].tolist()
assert names_reloaded == pipe_reloaded.get_feature_names()
print(f"[OK] Reloaded '{name}': shape {X_test_reloaded.shape}, finite, reproducible, "
      f"{len(names_reloaded)} aligned feature names.")

[OK] Reloaded 'ohe_topk': shape (175341, 63), finite, reproducible, 63 aligned feature names.


## 6. Summary

Chapter A data preparation is complete and verified. For each of the three
representations the notebook produced a training matrix and a testing matrix with
identical, aligned feature columns, all finite, and fitted only on the Chapter-A
training split. Reference labels are stored separately. **No clustering model was
trained.** The matrices in `data/processed/chapter_a/` are ready for Phase 4–6
(K-Means, CAH, evaluation) in later notebooks.

In [7]:
summary = [
    "# 03 - Chapter A data preparation summary",
    "",
    "Fitted on the official training split only; test transformed with the same pipelines.",
    "",
    "| representation | n_features | train | test | zero_frac(test) |",
    "|---|---|---|---|---|",
]
for name in representations:
    a = audit.loc[name]
    summary.append(f"| {name} | {int(a['n_features'])} | {int(a['train_rows'])} | "
                   f"{int(a['test_rows'])} | {a['test_zero_fraction']:.4f} |")
summary += [
    "",
    "- Features exclude id / label / attack_cat.",
    "- labels saved to y_train.csv / y_test.csv (separate from matrices).",
    "- fitted pipelines saved as pipeline_<rep>.joblib.",
    "- all matrices finite and column-aligned; round-trip verified.",
]
(REPORTS_DIR / "03_chapter_a_summary.md").write_text("\n".join(summary))
print("Saved ->", REPORTS_DIR / "03_chapter_a_summary.md")
print("\n=== Notebook 03 complete: Chapter A matrices prepared and verified. ===")

Saved -> /home/aminelby/Network-Traffic-Clustering/results/reports/03_chapter_a_summary.md

=== Notebook 03 complete: Chapter A matrices prepared and verified. ===
